In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, concat_ws, date_format, dayofmonth, dayofweek,
    lit, md5, month, quarter, row_number, weekofyear, when
)
from pyspark.sql.types import BooleanType, DateType, IntegerType
from pyspark.sql import Window
import pyspark.sql.functions as F
from datetime import date, timedelta

SRC_ACCOUNTS = "silver_accounts"
DIM_ACCOUNT  = "dim_account"     # final Gold table — no stg_ prefix
DIM_DATE     = "dim_date"        # final Gold table — no stg_ prefix

# Changes to any of these trigger a new SCD2 version
SCD2_TRACKED_COLS = [
    "customer_name", "email_masked", "phone_masked", "pan_masked",
    "age_bucket", "home_city", "branch", "account_open_date",
    "account_balance", "risk_category", "kyc_status", "account_status"
]
print("Config loaded.")

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 3, Finished, Available, Finished, False)

Config loaded.


In [2]:
df_incoming = (
    spark.table(SRC_ACCOUNTS)
    .withColumn(
        "record_hash",
        md5(concat_ws("|", *[col(c) for c in SCD2_TRACKED_COLS]))
    )
)
print(f"Incoming accounts: {df_incoming.count()}")
df_incoming.select("account_id", "risk_category", "kyc_status", "record_hash").show(5)

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 4, Finished, Available, Finished, False)

Incoming accounts: 500
+----------+-------------+----------+--------------------+
|account_id|risk_category|kyc_status|         record_hash|
+----------+-------------+----------+--------------------+
|      A101|          LOW|   PENDING|9f8340c9426dfe0a2...|
|      A102|       MEDIUM|  VERIFIED|0dd425f7f55da7b19...|
|      A103|       MEDIUM|   EXPIRED|198cf8c4c289b95ad...|
|      A104|       MEDIUM|  VERIFIED|6cfc4d5b0f2c78731...|
|      A105|          LOW|   EXPIRED|a6a68ce8dff4ec5c2...|
+----------+-------------+----------+--------------------+
only showing top 5 rows



In [3]:
today = date.today()

if spark.catalog.tableExists(DIM_ACCOUNT):

    df_existing    = spark.table(DIM_ACCOUNT)
    df_current_dim = df_existing.filter(col("is_current") == True)

    # Changed: same account_id but different hash
    df_changed = (
        df_incoming.alias("new")
        .join(
            df_current_dim.select("account_id", col("record_hash").alias("old_hash")).alias("old"),
            on="account_id", how="inner"
        )
        .filter(col("new.record_hash") != col("old_hash"))
        .select("new.*")
    )

    # Brand-new: account_id not in dim at all
    df_brand_new = df_incoming.join(
        df_current_dim.select("account_id"), on="account_id", how="left_anti"
    )

    df_to_insert  = df_changed.unionByName(df_brand_new)
    insert_count  = df_to_insert.count()
    changed_count = df_changed.count()
    new_count     = df_brand_new.count()

    print(f"Changed accounts   : {changed_count}")
    print(f"Brand-new accounts : {new_count}")
    print(f"Total to insert    : {insert_count}")

    if insert_count > 0:
        changed_ids = [r["account_id"] for r in df_changed.select("account_id").collect()]
        max_key     = df_existing.agg(F.max("account_key")).collect()[0][0] or 0

        # Close old records: is_current → False, valid_to → today
        df_existing_updated = (
            df_existing
            .withColumn("is_current",
                when(col("account_id").isin(changed_ids) & (col("is_current") == True),
                     lit(False)).otherwise(col("is_current")))
            .withColumn("valid_to",
                when(col("account_id").isin(changed_ids) & col("valid_to").isNull(),
                     lit(today).cast(DateType())).otherwise(col("valid_to")))
        )

        # New versions: new surrogate keys above current max
        window_seq = Window.orderBy(lit(1))
        df_new_versions = (
            df_to_insert
            .withColumn("_seq",       row_number().over(window_seq))
            .withColumn("account_key",(col("_seq") + max_key).cast("long"))
            .withColumn("is_current",  lit(True).cast(BooleanType()))
            .withColumn("valid_from",  lit(today).cast(DateType()))
            .withColumn("valid_to",    lit(None).cast(DateType()))
            .drop("_seq")
        )

        final_cols          = df_existing_updated.columns
        df_final_dim_account = (
            df_existing_updated
            .unionByName(df_new_versions.select(*final_cols))
        )
    else:
        print("No changes — dim_account is already up to date.")
        df_final_dim_account = df_existing

else:
    # First ever run
    print("First run — building dim_account from scratch.")
    window_seq = Window.orderBy(lit(1))
    df_final_dim_account = (
        df_incoming
        .withColumn("_seq",       row_number().over(window_seq))
        .withColumn("account_key", col("_seq").cast("long"))
        .withColumn("is_current",  lit(True).cast(BooleanType()))
        .withColumn("valid_from",  lit(today).cast(DateType()))
        .withColumn("valid_to",    lit(None).cast(DateType()))
        .drop("_seq")
    )

total = df_final_dim_account.count()
current_v = df_final_dim_account.filter(col("is_current") == True).count()
print(f"\ndim_account total rows     : {total}")
print(f"  Current versions         : {current_v}")
print(f"  Historical versions      : {total - current_v}")

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 5, Finished, Available, Finished, False)

Changed accounts   : 1
Brand-new accounts : 0
Total to insert    : 1

dim_account total rows     : 501
  Current versions         : 500
  Historical versions      : 1


In [4]:
df_final_dim_account.filter(col("account_id") == "A105").select(
    "account_key", "account_id", "risk_category",
    "is_current", "valid_from", "valid_to", "record_hash"
).orderBy("valid_from").show(truncate=False)
# Expected: 2 rows — Row 1 is_current=False, Row 2 risk_category=HIGH is_current=True

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 6, Finished, Available, Finished, False)

+-----------+----------+-------------+----------+----------+--------+--------------------------------+
|account_key|account_id|risk_category|is_current|valid_from|valid_to|record_hash                     |
+-----------+----------+-------------+----------+----------+--------+--------------------------------+
|5          |A105      |LOW          |true      |2026-06-19|NULL    |a6a68ce8dff4ec5c2e42a1fa0711b430|
+-----------+----------+-------------+----------+----------+--------+--------------------------------+



In [5]:
(
    df_final_dim_account.write
                        .mode("overwrite")
                        .format("delta")
                        .option("overwriteSchema", "true")
                        .saveAsTable(DIM_ACCOUNT)
)
print(f"Written: {DIM_ACCOUNT}  ({df_final_dim_account.count()} rows)")

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 7, Finished, Available, Finished, False)

Written: dim_account  (501 rows)


In [6]:
start_date, end_date = date(2026, 1, 1), date(2026, 12, 31)
date_list, d = [], start_date
while d <= end_date:
    date_list.append((d,))
    d += timedelta(days=1)

df_dim_date = (
    spark.createDataFrame(date_list, ["full_date"])
    .withColumn("full_date",    col("full_date").cast(DateType()))
    .withColumn("date_key",     (F.year("full_date") * 10000 +
                                 month("full_date") * 100 +
                                 dayofmonth("full_date")).cast(IntegerType()))
    .withColumn("day_of_week",  date_format("full_date", "EEEE"))
    .withColumn("day_of_month", dayofmonth("full_date"))
    .withColumn("week_of_year", weekofyear("full_date"))
    .withColumn("month_num",    month("full_date"))
    .withColumn("month_name",   date_format("full_date", "MMMM"))
    .withColumn("quarter",      quarter("full_date"))
    .withColumn("year",         F.year("full_date"))
    .withColumn("is_weekend",   dayofweek("full_date").isin([1, 7]))
    .select("date_key","full_date","day_of_week","day_of_month",
            "week_of_year","month_num","month_name","quarter","year","is_weekend")
)

(
    df_dim_date.write.mode("overwrite").format("delta")
               .option("overwriteSchema","true").saveAsTable(DIM_DATE)
)
print(f"Written: {DIM_DATE}  ({df_dim_date.count()} rows)")

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 8, Finished, Available, Finished, False)

Written: dim_date  (365 rows)


In [7]:
spark.stop()
print("Gold dims SCD2 run complete.")

StatementMeta(, e1562d07-d5b3-46f9-ab1b-3868f6aed173, 9, Finished, Available, Finished, False)

Gold dims SCD2 run complete.
